# 0b. Audit Output Tambahan

Melengkapi Fase 1: bikin `00_file_inventory.csv`, `01_schema_comparison.csv`, `02_data_quality.csv`, plus cek **duplikat penuh**, **distribusi kode**, **nilai ekstrem**, dan **konsentrasi penghapusan per tanggal** (buat Fase 2A).

Semua output ditulis ke `out/output_eda/`.

# 0. Import

In [1]:
import os
import pandas as pd
import numpy as np

# 1. Hitung Per Bulan

In [2]:
RAW = "../datasets/raw/2026"
valid_zone = set(range(1, 264))
valid_vendor = {1, 2, 6, 7}
valid_rate = {1, 2, 3, 4, 5, 6, 99}
valid_pay = {0, 1, 2, 3, 4, 5, 6}

inventory, quality, schemas = [], [], {}
konsentrasi = {}


def catat_tanggal(rule, dates):
    dd = konsentrasi.setdefault(rule, {})
    for k, v in pd.Series(dates).dt.date.value_counts().items():
        dd[k] = dd.get(k, 0) + int(v)


for m in range(1, 7):
    path = f"{RAW}/yellow_tripdata_2026-{m:02d}.parquet"
    d = pd.read_parquet(path)
    n = len(d)
    schemas[m] = dict(zip(d.columns, d.dtypes.astype(str)))

    inventory.append({
        "bulan": m, "file": os.path.basename(path),
        "ukuran_mb": round(os.path.getsize(path) / 1024 ** 2, 2),
        "jumlah_baris": n, "jumlah_kolom": d.shape[1],
        "pickup_min": d["tpep_pickup_datetime"].min(), "pickup_max": d["tpep_pickup_datetime"].max(),
        "dropoff_min": d["tpep_dropoff_datetime"].min(), "dropoff_max": d["tpep_dropoff_datetime"].max(),
    })

    dur = (d["tpep_dropoff_datetime"] - d["tpep_pickup_datetime"]).dt.total_seconds() / 60
    quality.append({
        "bulan": m, "jumlah_baris": n, "duplikat_penuh": int(d.duplicated().sum()),
        "pickup_di_luar_rentang": int(((d["tpep_pickup_datetime"] < "2026-01-01") | (d["tpep_pickup_datetime"] > "2026-06-30 23:59:59")).sum()),
        "dropoff_di_luar_rentang": int(((d["tpep_dropoff_datetime"] < "2026-01-01") | (d["tpep_dropoff_datetime"] > "2026-06-30 23:59:59")).sum()),
        "durasi_invalid": int(((dur <= 0) | (dur > 1440)).sum()),
        "dist_neg": int((d["trip_distance"] < 0).sum()),
        "dist_nol": int((d["trip_distance"] == 0).sum()),
        "dist_gt100": int((d["trip_distance"] > 100).sum()),
        "fare_neg": int((d["fare_amount"] < 0).sum()),
        "fare_gt500": int((d["fare_amount"] > 500).sum()),
        "total_neg": int((d["total_amount"] < 0).sum()),
        "total_gt500": int((d["total_amount"] > 500).sum()),
        "pass_nan": int(d["passenger_count"].isna().sum()),
        "pass_nol": int((d["passenger_count"] == 0).sum()),
        "pass_neg": int((d["passenger_count"] < 0).sum()),
        "pass_gt6": int((d["passenger_count"] > 6).sum()),
        "vendor_unknown": int((~d["VendorID"].isin(valid_vendor)).sum()),
        "ratecode_unknown": int((d["RatecodeID"].notna() & ~d["RatecodeID"].isin(valid_rate)).sum()),
        "payment_unknown": int((~d["payment_type"].isin(valid_pay)).sum()),
        "pu_unknown": int(((d["PULocationID"].isna()) | (~d["PULocationID"].isin(valid_zone))).sum()),
        "do_unknown": int(((d["DOLocationID"].isna()) | (~d["DOLocationID"].isin(valid_zone))).sum()),
    })

    # konsentrasi penghapusan per tanggal (urutan sama seperti cleaning)
    x = d.copy()
    m1 = (x["tpep_pickup_datetime"] < "2026-01-01") | (x["tpep_pickup_datetime"] > "2026-06-30 23:59:59")
    catat_tanggal("C1", x.loc[m1, "tpep_pickup_datetime"]); x = x[~m1]
    m2 = x["tpep_pickup_datetime"].isna() | x["tpep_dropoff_datetime"].isna()
    catat_tanggal("C2", x.loc[m2, "tpep_pickup_datetime"]); x = x[~m2]
    dur2 = (x["tpep_dropoff_datetime"] - x["tpep_pickup_datetime"]).dt.total_seconds() / 60
    m3 = (dur2 <= 0) | (dur2 > 1440)
    catat_tanggal("C3", x.loc[m3, "tpep_pickup_datetime"]); x = x[~m3]
    m4 = x["PULocationID"].isna() | (~x["PULocationID"].isin(valid_zone))
    catat_tanggal("C4", x.loc[m4, "tpep_pickup_datetime"]); x = x[~m4]
    m5 = x["DOLocationID"].isna() | (~x["DOLocationID"].isin(valid_zone))
    catat_tanggal("C5", x.loc[m5, "tpep_pickup_datetime"]); x = x[~m5]
    m6 = x.duplicated(keep="first")
    catat_tanggal("C6", x.loc[m6, "tpep_pickup_datetime"]); x = x[~m6]
    m7 = (x["trip_distance"] < 0) | (x["trip_distance"] > 100)
    catat_tanggal("C7", x.loc[m7, "tpep_pickup_datetime"]); x = x[~m7]
    m8 = (x["fare_amount"] < 0) | (x["fare_amount"] > 500) | (x["total_amount"] < 0) | (x["total_amount"] > 500)
    catat_tanggal("C8", x.loc[m8, "tpep_pickup_datetime"]); x = x[~m8]
    m9 = (x["passenger_count"] < 0) | (x["passenger_count"] > 6)
    catat_tanggal("C9", x.loc[m9, "tpep_pickup_datetime"]); x = x[~m9]
    m10 = (~x["VendorID"].isin(valid_vendor)) | (~x["payment_type"].isin(valid_pay)) | (x["RatecodeID"].notna() & ~x["RatecodeID"].isin(valid_rate))
    catat_tanggal("C10", x.loc[m10, "tpep_pickup_datetime"]); x = x[~m10]

    print(f"Bulan {m} selesai: {n:,} baris")

del d, x

Bulan 1 selesai: 3,724,889 baris
Bulan 2 selesai: 3,399,866 baris
Bulan 3 selesai: 3,952,451 baris
Bulan 4 selesai: 3,831,240 baris
Bulan 5 selesai: 4,090,836 baris
Bulan 6 selesai: 3,837,248 baris


# 2. Simpan 00 / 01 / 02

In [3]:
os.makedirs("../out/output_eda", exist_ok=True)

inventory_df = pd.DataFrame(inventory)
inventory_df.loc[len(inventory_df)] = {
    "bulan": "Total", "file": "", "ukuran_mb": round(inventory_df["ukuran_mb"].sum(), 2),
    "jumlah_baris": int(inventory_df["jumlah_baris"].sum()), "jumlah_kolom": "",
    "pickup_min": "", "pickup_max": "", "dropoff_min": "", "dropoff_max": ""}
inventory_df.to_csv("../out/output_eda/00_file_inventory.csv", index=False)
display(inventory_df)

# schema comparison
semua_kolom = []
for m in schemas:
    for c in schemas[m]:
        if c not in semua_kolom:
            semua_kolom.append(c)
baris = []
for c in semua_kolom:
    dtypes = {m: schemas[m].get(c, "-") for m in range(1, 7)}
    konsisten = len({v for v in dtypes.values() if v != "-"}) == 1
    baris.append({"kolom": c, **{f"bulan_{m}": dtypes[m] for m in range(1, 7)}, "konsisten": konsisten})
schema_df = pd.DataFrame(baris)
schema_df.to_csv("../out/output_eda/01_schema_comparison.csv", index=False)
display(schema_df)

quality_df = pd.DataFrame(quality)
quality_df.to_csv("../out/output_eda/02_data_quality.csv", index=False)
display(quality_df)

,bulan,file,ukuran_mb,jumlah_baris,jumlah_kolom,pickup_min,pickup_max,dropoff_min,dropoff_max
0,1,yellow_tripdata_2026-01.parquet,61.19,3724889,20,2025-12-31 23:57:29,2026-02-01 00:45:01,2025-12-31 23:57:32,2026-02-01 23:35:31
1,2,yellow_tripdata_2026-02.parquet,55.96,3399866,20,2026-01-31 23:31:23,2026-03-01 00:51:48,2026-01-31 23:36:52,2026-03-01 23:11:29
2,3,yellow_tripdata_2026-03.parquet,64.75,3952451,20,2008-12-31 23:03:20,2026-04-01 00:06:25,2009-01-01 00:07:36,2026-04-02 16:03:47
3,4,yellow_tripdata_2026-04.parquet,61.82,3831240,20,2001-01-01 09:23:58,2026-05-01 00:01:28,2001-01-01 16:09:38,2026-05-02 12:06:09
4,5,yellow_tripdata_2026-05.parquet,66.47,4090836,20,2008-12-31 23:05:53,2026-06-01 00:20:35,2009-01-01 17:08:07,2026-06-02 12:53:42
5,6,yellow_tripdata_2026-06.parquet,62.43,3837248,21,2008-12-31 23:03:25,2026-06-30 23:59:59,2009-01-01 14:28:05,2026-07-01 22:33:25
6,Total,,372.62,22836530,,,,,


,kolom,bulan_1,bulan_2,bulan_3,bulan_4,bulan_5,bulan_6,konsisten
0,VendorID,int32,int32,int32,int32,int32,int32,True
1,tpep_pickup_datetime,datetime64[us],datetime64[us],datetime64[us],datetime64[us],datetime64[us],datetime64[us],True
2,tpep_dropoff_datetime,datetime64[us],datetime64[us],datetime64[us],datetime64[us],datetime64[us],datetime64[us],True
3,passenger_count,float64,float64,float64,float64,float64,float64,True
4,trip_distance,float64,float64,float64,float64,float64,float64,True
5,RatecodeID,float64,float64,float64,float64,float64,float64,True
6,store_and_fwd_flag,str,str,str,str,str,str,True
7,PULocationID,int32,int32,int32,int32,int32,int32,True
8,DOLocationID,int32,int32,int32,int32,int32,int32,True
9,payment_type,int64,int64,int64,int64,int64,int64,True


,bulan,jumlah_baris,duplikat_penuh,pickup_di_luar_rentang,dropoff_di_luar_rentang,durasi_invalid,dist_neg,dist_nol,dist_gt100,fare_neg,...,total_gt500,pass_nan,pass_nol,pass_neg,pass_gt6,vendor_unknown,ratecode_unknown,payment_unknown,pu_unknown,do_unknown
0,1,3724889,0,6,2,45105,0,125738,162,39463,...,84,1088058,14787,0,7,0,0,0,5930,22302
1,2,3399866,0,0,0,40635,0,123250,172,26721,...,66,1023317,12884,0,4,0,0,0,5022,19347
2,3,3952451,0,2,2,49161,0,121210,161,20724,...,80,945748,12967,0,3,0,0,0,6249,20880
3,4,3831240,0,4,4,49532,0,93515,136,14328,...,84,799786,11692,0,4,0,0,0,6032,20470
4,5,4090836,0,2,2,52095,0,113031,136,14231,...,105,955371,12533,0,4,0,0,0,6486,21726
5,6,3837248,0,1,899,49838,0,128106,149,13648,...,114,1013500,10416,0,2,0,0,0,6569,23804


# 3. Konsentrasi Penghapusan per Tanggal

In [4]:
rows = []
for rule, dd in konsentrasi.items():
    total = sum(dd.values())
    if total == 0:
        rows.append({"aturan": rule, "total_dibuang": 0, "tanggal_paling_banyak": "-", "jumlah_di_tanggal_itu": 0, "%_di_tanggal_itu": 0.0})
        continue
    tgl, jml = max(dd.items(), key=lambda kv: kv[1])
    rows.append({"aturan": rule, "total_dibuang": total, "tanggal_paling_banyak": str(tgl),
                 "jumlah_di_tanggal_itu": jml, "%_di_tanggal_itu": round(jml / total * 100, 2)})

konsentrasi_df = pd.DataFrame(rows)
konsentrasi_df.to_csv("../out/output_eda/04_cleaning_concentration.csv", index=False)
display(konsentrasi_df)

,aturan,total_dibuang,tanggal_paling_banyak,jumlah_di_tanggal_itu,%_di_tanggal_itu
0,C1,15,2025-12-31,6,40.00
1,C2,0,-,0,0.00
2,C3,286366,2026-05-14,2145,0.75
3,C4,35923,2026-06-13,324,0.90
4,C5,108256,2026-06-14,949,0.88
5,C6,0,-,0,0.00
6,C7,616,2026-02-26,9,1.46
7,C8,130786,2026-01-03,1480,1.13
8,C9,18,2026-01-03,1,5.56
9,C10,0,-,0,0.00


# 4. Cek Koordinat di Wilayah NYC

In [5]:
zona = pd.read_csv("../datasets/zone/zone_centroid_lookup.csv")
# bounding box kasar NYC
lat_min, lat_max = 40.4, 41.0
lon_min, lon_max = -74.3, -73.6
luar = zona[(zona["latitude"] < lat_min) | (zona["latitude"] > lat_max) |
            (zona["longitude"] < lon_min) | (zona["longitude"] > lon_max)]
print("Total zona:", len(zona))
print("Zona di luar bounding box NYC:", len(luar))
print("Latitude range:", zona["latitude"].min(), "-", zona["latitude"].max())
print("Longitude range:", zona["longitude"].min(), "-", zona["longitude"].max())
if len(luar):
    display(luar[["LocationID", "zone_name", "borough", "latitude", "longitude"]])

Total zona: 263
Zona di luar bounding box NYC: 0
Latitude range: 40.52549458176382 - 40.89952906437648
Longitude range: -74.2335335493765 - -73.71102598691236


## Catatan

- `duplikat_penuh` dihitung pakai `DataFrame.duplicated()` (semua kolom).
- Nilai ekstrem pakai batas kasar (jarak > 100 mil, tarif > $500) yang sama dengan aturan cleaning.
- `%_di_tanggal_itu` = porsi baris yang dibuang pada satu tanggal dengan penghapusan terbanyak. Kalau kecil, penghapusan tidak terkonsentrasi di satu tanggal.